# LeapQuant 教程 (5/5): 运行时接线 —— vllm-ascend 里 LeapQuant 怎么跑起来

前四本是数值; 这本是工程。PR1 (#21120) 在 vllm-ascend 里改了哪些文件、每步谁调用谁:

| 文件 | 干什么 |
|---|---|
| `ascend_config.py` | `LeapQuantConfig` (window / compensator_tokens + 互斥项) |
| `mamba_state/leapquant/state.py` | 数值本体 (教程 3 的四个函数 + due 判定 mask) |
| `mamba_state/leapquant/runtime.py` | `leapquant_flush_state_rows`: 把 flush 应用到状态池的行 |
| `ops/gdn.py` | GDN forward 末尾的 flush 挂接点 |
| `ops/gdn_attn_builder.py` / `kimi_kda.py` | 给请求挂 LeapQuant 元数据 |
| `tests/.../a3_2/` | A3 实卡 UT (锚点: 输出漂移 0.37% / 状态漂移 0.53%) |

逐行讲解见仓库 `TORCH_NPU_WALKTHROUGH.md`。本 notebook 动手演示其中三块:
**配置 → due 判定 → 池级行 flush**, 最后跑一遍 64-token 集成复验。

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发树 (分支 leapquant-triton-m3)

try:
    import torch_npu  # noqa: F401 — 注册 npu 后端
    DEV = torch.device("npu") if torch.npu.is_available() else torch.device("cpu")
except Exception:
    DEV = torch.device("cpu")

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("vllm").setLevel(logging.ERROR)

torch.manual_seed(20261110)
print(f"torch = {torch.__version__}   device = {DEV}")

torch = 2.10.0+cpu   device = npu


/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


In [2]:
H, K, V = 8, 128, 128        # Qwen3.5-0.8B 单卡 GDN 头几何 (128×128)
B, WINDOW, R = 4, 16, 4      # 批 / 窗口 / 补偿秩 (论文 8-bit 最优)
SCALE = K ** -0.5

def gdn_inputs(B, H, T, K=K, V=V, dev=DEV):
    q = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=dev), dim=-1)
    k = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=dev), dim=-1)
    v = torch.randn(B, H, T, V, device=dev)
    g = -torch.rand(B, H, T, device=dev) * 0.1       # log 域衰减率 ∈ [-0.1, 0)
    beta = torch.rand(B, H, T, device=dev).sigmoid() # delta 门控
    return q, k, v, g, beta

def gdn_step(h, q, k, v, g, beta):
    """一步递推 (逐项拆解见教程 1): 衰减 → delta 修正 → 外积更新。"""
    h = h * g.exp()[..., None, None]
    v_adj = (v - torch.einsum("bhvk,bhk->bhv", h, k)) * beta[..., None]
    return h + torch.einsum("bhv,bhk->bhvk", v_adj, k)

def gdn_out(h, q):
    return torch.einsum("bhvk,bhk->bhv", h, q) * SCALE

def trajectory(inputs, window=None, boundary_fn=None, every_step_fn=None):
    """跑 T 步; 可选在窗口边界 / 每步对状态做一次变换 (量化实验的注入点)。"""
    q, k, v, g, beta = inputs
    B, H, T = q.shape[0], q.shape[1], q.shape[2]
    h = torch.zeros(B, H, V, K, device=q.device)
    outs = []
    for t in range(T):
        h = gdn_step(h, q[:, :, t], k[:, :, t], v[:, :, t], g[:, :, t], beta[:, :, t])
        if every_step_fn is not None:
            h = every_step_fn(h)
        elif boundary_fn is not None and (t + 1) % window == 0:
            h = boundary_fn(h)
        outs.append(gdn_out(h, q[:, :, t]))
    return h, torch.stack(outs, dim=2)

print("GDN 模拟器就绪 (完整拆解见教程 1)")

GDN 模拟器就绪 (完整拆解见教程 1)


## 1. 配置: `LeapQuantConfig`

引擎侧只暴露两个数值旋钮 (就是教程 4 扫的那两个), 其余是实现开关;
它与 cudagraph/某些加速特性互斥 (量化 flush 需要逐步执行, 与整图捕获不兼容)。

In [3]:
from vllm_ascend.ascend_config import LeapQuantConfig

cfg = LeapQuantConfig(enabled=True, window=16, compensator_tokens=4)
print(f"window = {cfg.window}          ← 每 16 个 token 拍一次 int8 快照")
print(f"compensator_tokens = {cfg.compensator_tokens}  ← fp16 补偿方向数 r")
print("完整定义见 vllm_ascend/ascend_config.py (与 FA / cudagraph 等的互斥项也在那里)")

INFO 11-10 19:08:58 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-10 19:08:58 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-10 19:08:58 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-10 19:08:58 [__init__.py:272] Platform plugin ascend is activated


INFO 11-10 19:08:59 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


window = 16          ← 每 16 个 token 拍一次 int8 快照
compensator_tokens = 4  ← fp16 补偿方向数 r
完整定义见 vllm_ascend/ascend_config.py (与 FA / cudagraph 等的互斥项也在那里)


## 2. due 判定: 这一步之后**哪些请求**该 flush

池里同时有很多请求, 各自进度不同 —— 只有序列长度**恰好跨过窗口边界**的行才 due。
两个 mask 的分工:

- `decode_flush_mask(processed, w)`: 解码行, 每步 +1, 步末恰好落在边界才 due;
- `crossed_boundary_mask(seq_len, query_len, w)`: prefill chunk 行, chunk 可能**一步跨多个 token**,
  只要跨过 (哪怕越过) 内部边界就 due —— 这是 M3.3 的跨界语义。

In [4]:
from vllm_ascend.quantization.methods.mamba_state.leapquant.state import (
    decode_flush_mask, crossed_boundary_mask,
)

seq  = torch.tensor([16, 17, 33])      # 三个请求的当前序列长度
qlen = torch.tensor([1, 1, 12])        # 本步各自处理的 token 数
print("crossed_boundary_mask:", crossed_boundary_mask(seq, qlen, 16).tolist(),
      " ← 16 步末恰在边界✓; 17 不 due; 33 跨过 32 (chunk 内部边界)✓")
print("decode_flush_mask   :", decode_flush_mask(seq - qlen, 16).tolist(),
      " ← 解码语义: 本步处理前恰好停在边界 (16-1=15 不 due, 17-1=16 due)")

crossed_boundary_mask: [True, False, True]  ← 16 步末恰在边界✓; 17 不 due; 33 跨过 32 (chunk 内部边界)✓
decode_flush_mask   : [True, False, False]  ← 解码语义: 本步处理前恰好停在边界 (16-1=15 不 due, 17-1=16 due)


## 3. 池级行 flush: `leapquant_flush_state_rows`

状态池是 `[SLOTS, H, V, K]` 的大张量, 每个并发请求占一行 (槽位)。flush 只允许碰
**due 的行** —— 其余行 (包括批外槽) 必须逐位不动, 否则会污染别人的状态。

In [5]:
from vllm_ascend.quantization.methods.mamba_state.leapquant.runtime import (
    leapquant_flush_state_rows,
    leapquant_config_or_none,
)
from vllm_ascend.quantization.methods.mamba_state.leapquant.state import leap_window_flush

SLOTS = 8
pool = torch.randn(SLOTS, H, V, K, device=DEV) * 0.1
idx  = torch.tensor([3, 1, 5, 7], device=DEV)     # 本批 4 个请求占的槽位
seq_lens = torch.tensor([16, 23, 32, 41])         # → 槽 3 (seq=16) 与槽 5 (seq=32) due
before = pool.clone()
leapquant_flush_state_rows(pool, idx, seq_lens, config=cfg)

due_slots = idx[[i for i, s in enumerate(seq_lens.tolist()) if s % cfg.window == 0]]
want_batched = leap_window_flush(before[due_slots])   # 与 flush 相同的批内 N, 排除批大小的 QR 舍入差
due_list = due_slots.tolist()
for j, r_ in enumerate(idx.tolist()):
    s_ = int(seq_lens[j])
    if s_ % cfg.window == 0:
        wj = due_list.index(r_)
        ok = torch.allclose(pool[r_], want_batched[wj], atol=1e-5)
        print(f"slot {r_} (seq={s_}, due):  == leap_window_flush(行): {ok}"
              f"   max|Δ| = {(pool[r_] - want_batched[wj]).abs().max():.1e}")
    else:
        print(f"slot {r_} (seq={s_}, 非due): 逐位未动: {torch.equal(pool[r_], before[r_])}")
untouched = [i for i in range(SLOTS) if i not in idx.tolist()]
print("批外槽逐位未动:", all(torch.equal(pool[i], before[i]) for i in untouched))

[W1110 19:09:05.446204074 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


slot 3 (seq=16, due):  == leap_window_flush(行): True   max|Δ| = 0.0e+00
slot 1 (seq=23, 非due): 逐位未动: True
slot 5 (seq=32, due):  == leap_window_flush(行): True   max|Δ| = 0.0e+00
slot 7 (seq=41, 非due): 逐位未动: True
批外槽逐位未动: True


## 4. 挂接点: 一次 forward 的调用链

```text
请求进入 → gdn_attn_builder / kimi_kda:
              给请求挂 LeapQuant 元数据 (窗口计数等)
GDN forward (ops/gdn.py):
              ① 正常递推 (prefill chunk / decode 步)
              ② forward 末尾: crossed_boundary_mask / decode_flush_mask 找出 due 行
              ③ leapquant_flush_state_rows(pool, due 行)   ← 教程 3 整条流水线
MambaPool:    状态池的分配 / 槽位生命周期
```

也就是: **数值 (教程 3) + 节奏 (教程 4) + 挂接点 (本节)** = PR1 全部。

## 5. 集成规模复验: 64 token / 4 次 flush

对应 A3 实卡集成 UT (`test_leapquant_gdn_integration.py`, 真机 AscendC 递推算子) 的锚点
**输出漂移 0.37% / 状态漂移 0.53%**。这里在模拟器上跑同一规模对照量级。

In [6]:
torch.manual_seed(20261110)               # 本 cell 独立定种, 结果与执行历史无关
T_INT, B1 = 64, 3
inp = gdn_inputs(B1, H, T_INT)
h_r, o_r = trajectory(inp)
h_l, o_l = trajectory(inp, window=WINDOW, boundary_fn=leap_window_flush)

out_drift = ((o_l - o_r).norm() / o_r.norm()).item()
st_drift  = ((h_l - h_r).norm() / h_r.norm()).item()
print(f"64 token / window=16 / r=4:  输出漂移 {out_drift:.2%}   状态漂移 {st_drift:.2%}")
print("对照 A3 实卡 UT 锚点:        输出 0.37%   状态 0.53%  (同一量级)")

64 token / window=16 / r=4:  输出漂移 0.34%   状态漂移 0.56%
对照 A3 实卡 UT 锚点:        输出 0.37%   状态 0.53%  (同一量级)


## 小结 (教程 5) —— 系列完

- `LeapQuantConfig` 只露 window / r 两个旋钮;
- due 判定区分 decode 行 (步末恰好边界) 与 prefill chunk 行 (跨界即 due);
- 池级 flush 只碰 due 行, 其余逐位不动;
- 模拟器 64-token 漂移与 A3 实卡 UT 锚点同量级。

下一步:
- **逐行讲解** → 仓库 `TORCH_NPU_WALKTHROUGH.md`;
- **triton 融合内核** (PR2: flush 18× 提速 / payload 池 3.37× 压缩 / decode 内核) →
  `leapquant_triton_npu.ipynb`。